# Formatting and cleaning

## 1. Load basic modules and the csv file 

In [1]:
import pandas as pd
import numpy as np
import random as rd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder, StandardScaler


data = pd.read_csv('../data_raw/train.csv')
data.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response
0,1,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1
1,2,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0
2,3,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1
3,4,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0
4,5,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0


## 2. Checking Dataset Shape,  null Values and feature types

**Fixing incorrect, incomplete, duplicate or otherwise erroneous data**

In [ ]:
data.shape

In [ ]:
data.isnull().sum()

In [13]:
data.dtypes
data.drop("id", axis = 1)

,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,Response,log_premium
0,Male,44,1,28.0,0,> 2 Years,Yes,40454.0,26.0,217,1,10.607921
1,Male,76,1,3.0,0,1-2 Year,No,33536.0,26.0,183,0,10.420375
2,Male,47,1,28.0,0,> 2 Years,Yes,38294.0,26.0,27,1,10.553049
3,Male,21,1,11.0,1,< 1 Year,No,28619.0,152.0,203,0,10.261826
4,Female,29,1,41.0,1,< 1 Year,No,27496.0,152.0,39,0,10.221796
...,...,...,...,...,...,...,...,...,...,...,...,...
381104,Male,74,1,26.0,1,1-2 Year,No,30170.0,26.0,88,0,10.314603
381105,Male,30,1,37.0,1,< 1 Year,No,40016.0,152.0,131,0,10.597035
381106,Male,21,1,30.0,1,< 1 Year,No,35118.0,160.0,161,0,10.466469
381107,Female,68,1,14.0,0,> 2 Years,Yes,44617.0,other,74,0,10.705870


**As we can see, there is no null data in the dataset. We drop columns 'id' because it doesnt provide relevant information for our scope**


## 3. Importing functions from Cleaning_functions.ipynb

In [3]:
import sys
sys.path.append('../src/')
import Cleaning_functions as cf

### 3.1 Continous variables

**Removing outliers from the "Annual_Premium" column**. 

The original data are found with a large scatter, outliers are interpreted as errors in data entry. Given the distribution, it is decided to use log transformation. Log transformation  de-emphasizes outliers and allows us to potentially obtain a bell-shaped distribution. The idea is that taking the log of the data can restore symmetry to the data.

**Data with a high dispersion interpreted as erroneous are eliminated. The dataset is reduced by 80172 rows (21%).**

In [4]:
data = cf.correct_Annual_Premium(data)

In [8]:
data.shape

(300937, 13)

**Changing values from the "Policy_Sales_Channel" column**

This columns has over 148 different values, we grouped into 'others' category the values that are less than 2%

In [9]:
len(data['Policy_Sales_Channel'].unique())

148

In [10]:
data['Policy_Sales_Channel'].value_counts(normalize=True)

152.0    0.377272
26.0     0.218155
124.0    0.199364
160.0    0.050143
122.0    0.028135
           ...   
43.0     0.000003
84.0     0.000003
97.0     0.000003
2.0      0.000003
99.0     0.000003
Name: Policy_Sales_Channel, Length: 148, dtype: float64

In [11]:
data = cf.correct_policy(data)



In [12]:
data.Policy_Sales_Channel.value_counts(normalize=True)

152.0    0.377272
other    0.326294
26.0     0.218155
160.0    0.050143
122.0    0.028135
Name: Policy_Sales_Channel, dtype: float64

## Dummy variables, train/test and standardization

As we have categorical variables, we need to turn them to dummies variables

In [ ]:
categoricals = ["Gender", "Driving_License", "Region_Code", "Previously_Insured", "Vehicle_Age", "Policy_Sales_Channel", 
               ]

enc = OneHotEncoder(drop = "first")
X = data[categoricals]
enc.fit(X)
enc.categories_

In [ ]:
dummies = enc.transform(X).toarray()

In [ ]:
dummies_df = pd.DataFrame(dummies)

In [ ]:
#col_names = [categoricals[i] + '_' + enc.categories_[i] for i in range(len(categoricals))] 